ERP Analysis Procedure**

### **1. Load Raw EEG File**

* Load the `.set` EEG file for a participant using MNE.

---

### **2. Re-reference**

* Re-reference EEG data to the **average of LM (Left Mastoid)** and **RM (Right Mastoid)** **before any other step**.

---

### **3. ICA for Blink Artifact Removal**

* Run **ICA on the unfiltered or minimally filtered data** (preferably **raw or lightly filtered**, e.g., 0.01–50 Hz).
* Use **all 128 channels** (not just 81).
* **Plot components** to visually inspect and **identify blink-related components**.

  * These are typically **among the first 5** due to high amplitude.
* **Manually exclude blink components only** (avoid over-correction).

---

### **4. Apply ICA**

* Apply ICA solution to the raw data.
* Visually **inspect ICA-corrected signal** (before filtering).

---

### **5. Interpolate Bad Channels**

* Identify bad/noisy channels (especially **central anterior/medial** regions).
* **Interpolate them using weighted interpolation**.
* Try to limit exclusions; preserve as much spatial coverage as possible.

---

### **6. Filter**

* Apply band-pass filter **from 0.01 Hz to 50 Hz**.

  * Can be done before or after ICA depending on blink removal performance.
* Apply **notch filter at 60 Hz** to remove line noise.

---

### **7. Extract Events**

* Extract events from annotations:

  * `Wd2E` = normal context
  * `Wd2N` = slowed context

---

### **8. Create Epochs**

* Epochs should be **time-locked to event markers** (`Wd2E`/`Wd2N`).
* Use **epoch window**: -100 ms to 800 ms.
* Apply **baseline correction** using the **100 ms pre-onset period**.

---

### **9. Reject Artifactual Epochs**

* Automatically or manually reject epochs with high-amplitude noise or eye movement remnants.
* Use voltage thresholds, peak-to-peak rejection, or visual inspection.

---

### **10. Save Cleaned Epochs**

* Save the cleaned and processed epochs per subject (`.fif` format).

---

### **11. Compute Subject-level ERPs**

* For each subject, compute average ERPs:

  * ERP for all `Wd2E` epochs (normal context)
  * ERP for all `Wd2N` epochs (slowed context)

---

### **12. Grand Average (Group-Level ERPs)**

* Average across all 21 participants:

  * Grand average ERP for `Wd2E`
  * Grand average ERP for `Wd2N`

---

### **13. Plot ERPs (9 Scalp Regions)**

* Use all **128 channels** and group into 9 regions:

  * Left/Medial/Right × Anterior/Central/Posterior
* Plot ERP waveforms over time (−100 ms to 400 ms) showing both conditions.
* Highlight **N100** and **N280** components.




In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
import mne
import os
import numpy as np
from scipy.signal import hilbert
import matplotlib.pyplot as plt

In [ ]:
# Set path
root_dir = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

# Process one participant
def load_and_reference(participant):
    folder_path = os.path.join(root_dir, participant)
    set_files = [f for f in os.listdir(folder_path) if f.endswith('.set')]

    if not set_files:
        print(f"No .set file found in {participant}")
        return None

    raw_path = os.path.join(folder_path, set_files[0])
    raw = mne.io.read_raw_eeglab(raw_path, preload=True)

    # Average LM (E57) and RM (E100) for re-referencing
    try:
        lm = raw.copy().pick_channels(['E57']).get_data()
        rm = raw.copy().pick_channels(['E100']).get_data()
        mastoid_avg = (lm + rm) / 2
        raw._data = raw.get_data() - mastoid_avg  # apply linked mastoid referencing
        print(f"Re-referenced {participant} using LM/RM average.")
        return raw

    except Exception as e:
        print(f"Error processing {participant}: {str(e)}")
        return None

# Loop through all participants
raw_dict = {}
for pid in participants:
    raw = load_and_reference(pid)
    if raw:
        raw_dict[pid] = raw